# Linguagens e Ambientes de Programação
## (Aula Teórica 11)

### Agenda

- Arvores n-árias

## Árvores n-árias

Numa árvore n-ária (`ntree`) cada nó pode ter **qualquer número de filhos**, representados como uma lista de sub-árvores. Considere o tipo indutivo definido abaixo e o o exemplo `nt` — uma árvore com raiz 1, subarvores com raizes 2, 4 e 5, e o sendo que a subarvore com o valor 2 na raiz tem uma subarvore com o valor 3.

In [6]:
type 'a ntree = NEmpty | NNode of 'a * 'a ntree list

let nt = NNode (1, [NNode (2, [NNode (3, [])]); NNode (4, []); NNode(5,[])])

type 'a ntree = NEmpty | NNode of 'a * 'a ntree list


val nt : int ntree =
  NNode (1, [NNode (2, [NNode (3, [])]); NNode (4, []); NNode (5, [])])


Esta outra definição de tipos é mais precisa no sentido em que não há confusão entre usar uma arvore vazia nas subarvores ou uma lista vazia. Vai requerer sempre que todas as funções tenham os casos para tratar a arvore vazia no nível de topo.

In [ ]:
type 'a netree = NNode of 'a * 'a netree list 

type 'a nntree = NNEmpty | NNNEmpty of 'a netree

type 'a netree = NNode of 'a * 'a netree list


type 'a nntree = NNEmpty | NNNEmpty of 'a netree


### Soma de todos os valores numa árvore n-ária — versão 1 (com função auxiliar explícita)

`sum_tree` soma todos os valores da árvore. A função auxiliar `sum_aux` itera sobre a lista de filhos, chamando `sum_tree` em cada sub-árvore.

In [9]:
let rec sum_tree t = 
  let rec sum_aux l = 
    match l with 
    | [] -> 0
    | t'::ts -> sum_tree t' + sum_aux ts
  in 
  match t with 
  | NEmpty -> 0
  | NNode(v, l) -> v + sum_aux l 

let _ = assert (sum_tree nt = 15)

val sum_tree : int ntree -> int = <fun>


- : unit = ()


### Soma dos valores — versão 2 (com `fold_left`)

Versão mais compacta que substitui a auxiliar explícita por `List.fold_left`. Nota: o acumulador inicial é `v` (o valor do nó atual), o que pode ser confuso — compare com a versão 4.

In [ ]:
let rec sum_tree t = 
  match t with 
  | NEmpty -> 0
  | NNode(v, l) -> List.fold_left (fun acc t' -> acc + sum_tree t') v l 

let _ = assert (sum_tree nt = 15)

val sum_tree : int ntree -> int = <fun>


- : unit = ()


val sum_tree : int ntree -> int = <fun>


- : unit = ()


Podemos ainda utilizar uma forma mais simples, iterando primeiro a lista de subarvores e depois somando os valores recolhidos.

In [12]:
let rec sum_tree t = 
  match t with 
  | NEmpty -> 0
  | NNode(v, l) -> List.map sum_tree l |> List.fold_left (+) v  

let _ = assert (sum_tree nt = 15)

val sum_tree : int ntree -> int = <fun>


- : unit = ()


### Mapeamento sobre árvores n-árias — versão 1

`map_ntree f t` aplica `f` a cada valor da árvore preservando a estrutura. Para cada nó, aplica `f` ao valor e mapeia recursivamente sobre a lista de filhos usando uma lambda explícita `(fun t' -> map_ntree f t')`.

In [14]:
let rec map_ntree f nt = 
  match nt with
  | NEmpty -> NEmpty
  | NNode (v, l) -> NNode (f v, List.map (fun t' -> map_ntree f t') l)

val map_ntree : ('a -> 'b) -> 'a ntree -> 'b ntree = <fun>


In [18]:
let _ = assert (map_ntree ((+)1) nt = NNode (2, [NNode (3, [NNode (4, [])]); NNode (5, []); NNode (6, [])]))

- : unit = ()


### Mapeamento — versão 2 (estilo point-free)

Versão mais concisa onde a função anónima `fun t' -> map_ntree f t'` passa a ser `map_ntree f` que denota a função que espera o argumento que representa a árvore. 

In [42]:
let rec map_ntree f nt = 
  match nt with
  | NEmpty -> NEmpty
  | NNode (v, l) -> NNode (f v, List.map (map_ntree f) l)

val map_ntree : ('a -> 'b) -> 'a ntree -> 'b ntree = <fun>


In [19]:
let _ = assert (map_ntree ((+)1) nt = NNode (2, [NNode (3, [NNode (4, [])]); NNode (5, []); NNode (6, [])]))

- : unit = ()


### Fold seguindo a pré-ordem sobre árvores n-árias — versão 1 (função auxiliar recursiva)

`prefix_fold_ntree f acc t` percorre a árvore usando **pré-ordem** (raiz antes das subárvores). 
Variante com a função auxiliar `fold_aux` que itera manualmente sobre a lista de filhos, sem usar `List.fold_left`. Torna o padrão de acumulação mais visível.

In [34]:
let rec prefix_fold_ntree f acc nt = 
  let rec fold_aux acc l = 
    match l with 
    | [] -> acc
    | nt'::nts -> let acc_nt' = prefix_fold_ntree f acc nt' in fold_aux acc_nt' nts 
  in 
  match nt with
  | NEmpty -> acc
  | NNode (v, l) -> let acc_v = f acc v in fold_aux acc_v l
  
let _ = assert (prefix_fold_ntree (+) 0 nt = 15)
let _ = assert (prefix_fold_ntree (fun acc x -> acc@[x]) [] nt = [1; 2; 3; 4; 5])
let _ = assert (List.rev (prefix_fold_ntree (fun acc x -> x::acc) [] nt) = [1;2;3;4;5])

val prefix_fold_ntree : ('a -> 'b -> 'a) -> 'a -> 'b ntree -> 'a = <fun>


- : unit = ()


- : unit = ()


- : unit = ()


### Fold em pré-ordem — versão 2 (com fold_left)
Esta versão usa fold_left para percorrer a lista de subarvores e propaga o resultado com um parametro acumulador. Aplica `f acc v` ao valor da raiz, regista o resultado em `acc_v` e depois percorre a lista das subárvores com `fold_left` passando `acc_v` como valor inicial do acumulador. Não é possível fazer o padrão map seguido de fold porque o processamento de cada uma das subarvores depende dos valores anteriores.

In [33]:
let rec prefix_fold_ntree f acc nt = 
  match nt with
  | NEmpty -> acc
  | NNode (v, l) -> 
    let acc_v = f acc v in 
    List.fold_left (fun acc nt' -> prefix_fold_ntree f acc nt') acc_v l

let _ = assert (prefix_fold_ntree (+) 0 nt = 15)
let _ = assert (prefix_fold_ntree (fun acc x -> acc@[x]) [] nt = [1; 2; 3; 4; 5])
let _ = assert (List.rev (prefix_fold_ntree (fun acc x -> x::acc) [] nt) = [1;2;3;4;5])

val prefix_fold_ntree : ('a -> 'b -> 'a) -> 'a -> 'b ntree -> 'a = <fun>


- : unit = ()


- : unit = ()


- : unit = ()


### Fold em pré-ordem — versão 2 (point-free)

Versão mais compacta com avaliação parcial da função usada no fold:

In [32]:
let rec prefix_fold_ntree f acc nt = 
  match nt with
  | NEmpty -> acc
  | NNode (v, l) -> let acc_v = f acc v in List.fold_left (prefix_fold_ntree f) acc_v l

let _ = assert (prefix_fold_ntree (+) 0 nt = 15)
let _ = assert (prefix_fold_ntree (fun acc x -> acc@[x]) [] nt = [1; 2; 3; 4; 5])
let _ = assert (List.rev (prefix_fold_ntree (fun acc x -> x::acc) [] nt) = [1;2;3;4;5])

val prefix_fold_ntree : ('a -> 'b -> 'a) -> 'a -> 'b ntree -> 'a = <fun>


- : unit = ()


- : unit = ()


- : unit = ()


### Construir uma BST a partir de uma árvore n-ária

Aplica `insert` como função de fold: percorre `nt` em pré-ordem e insere cada valor numa BST inicialmente vazia (`Leaf`). Demonstra que `prefix_fold_ntree` é suficientemente geral para combinar tipos de dados diferentes.

In [42]:
type 'a tree = Empty | Node of 'a * 'a tree * 'a tree

let rec insert_bst x t = 
  match t with 
  | Empty -> Node(x,Empty,Empty)
  | Node(y,l,r) -> if x <= y then Node(y,insert_bst x l,r) else Node(y,l,insert_bst x r)

let _ = prefix_fold_ntree (fun bst x -> insert_bst x bst) Empty nt


let flip f = fun x y -> f y x 
let _ = prefix_fold_ntree (flip insert_bst) Empty nt

type 'a tree = Empty | Node of 'a * 'a tree * 'a tree


val insert_bst : 'a -> 'a tree -> 'a tree = <fun>


- : int tree =
Node (1, Empty,
 Node (2, Empty, Node (3, Empty, Node (4, Empty, Node (5, Empty, Empty)))))


val flip : ('a -> 'b -> 'c) -> 'b -> 'a -> 'c = <fun>


- : int tree =
Node (1, Empty,
 Node (2, Empty, Node (3, Empty, Node (4, Empty, Node (5, Empty, Empty)))))
